# Brazil 2026 — Who Votes: Age, Gender and Schooling
### The electorate behind the result

The three previous views treated the electorate as a number. This notebook opens it up. Brazil's 2026 register — **158.7 million voters**, published by the TSE in July 2026 — records every voter by municipality, age bracket, gender, schooling and marital status. Joined to the results, it lets us ask three questions:

1. **How old is the Brazilian electorate**, and how does that differ across states and municipalities?
2. **Does the age structure of a municipality predict how it voted?**
3. **Does it predict who turns out?**

The answers are asymmetric, and that asymmetry is the main finding: **demography strongly predicts the vote and almost not at all the turnout**.

A warning that applies to every correlation below: this is **ecological analysis** — municipalities, not individuals. A correlation between the median age of a municipality and its vote does not mean that older voters voted for Bolsonaro; it means that municipalities with older electorates tended to. Notebook-level claims are stated with that caveat in mind.

In [16]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# locate the project root (works whichever directory Jupyter was started from)
_p = Path.cwd()
while not (_p / "00_data_preprocessing_validation.ipynb").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import statsmodels.formula.api as smf
from IPython.display import HTML, display

import types, nbformat

_lib_path = _p / "00_data_preprocessing_validation.ipynb"
_lib_src = "\n".join(
    c.source for c in nbformat.read(str(_lib_path), as_version=4).cells
    if c.cell_type == "code" and "library" in (c.metadata.get("tags") or [])
)
elib = types.ModuleType("elib")
elib.__dict__["__file__"] = str(_lib_path)
exec(compile(_lib_src, str(_lib_path), "exec"), elib.__dict__)

if getattr(elib, "LIB_VERSION", 0) < 2 or not hasattr(elib, "enable_plotting"):
    print("WARNING: 00_data_preprocessing_validation.ipynb is older than this notebook.")
    print("         Copy the whole repository again (all four notebooks together).")
    try:
        import nbformat  # noqa: F401
        pio.renderers.default = "plotly_mimetype"
    except Exception:
        pio.renderers.default = "notebook_connected"
else:
    elib.enable_plotting()
pd.set_option("display.width", 170)
con = elib.db("2GB")
print("ready")

ready


In [17]:
# age brackets -> age interval (CD_FAIXA_ETARIA codes used by the TSE)
BRACKETS = [(1600, 16, 16), (1700, 17, 17), (1800, 18, 18), (1900, 19, 19), (2000, 20, 20),
            (2124, 21, 24), (2529, 25, 29), (3034, 30, 34), (3539, 35, 39), (4044, 40, 44),
            (4549, 45, 49), (5054, 50, 54), (5559, 55, 59), (6064, 60, 64), (6569, 65, 69),
            (7074, 70, 74), (7579, 75, 79), (8084, 80, 84), (8589, 85, 89), (9094, 90, 94),
            (9599, 95, 99), (9999, 100, 110)]
LO = {b[0]: b[1] for b in BRACKETS}
HI = {b[0]: b[2] for b in BRACKETS}
MID = {k: (LO[k] + HI[k]) / 2 for k in LO}
LABEL = {1600: "16", 1700: "17", 1800: "18", 1900: "19", 2000: "20", 2124: "21-24",
         2529: "25-29", 3034: "30-34", 3539: "35-39", 4044: "40-44", 4549: "45-49",
         5054: "50-54", 5559: "55-59", 6064: "60-64", 6569: "65-69", 7074: "70-74",
         7579: "75-79", 8084: "80-84", 8589: "85-89", 9094: "90-94", 9599: "95-99", 9999: "100+"}

def weighted_median(d: pd.DataFrame) -> float:
    # Median age from bracket counts, interpolating inside the median bracket.
    d = d[d.CD_FX.isin(LO)].sort_values("CD_FX")
    if d.ELEITORES.sum() == 0:
        return np.nan
    cum = d.ELEITORES.cumsum() / d.ELEITORES.sum()
    i = int(cum.searchsorted(0.5))
    i = min(i, len(d) - 1)
    if i == 0:
        return MID[d.CD_FX.iloc[0]]
    prev, cur = d.iloc[i - 1], d.iloc[i]
    lo_prev, hi_cur = LO[prev.CD_FX], HI[cur.CD_FX]
    span = cum.iloc[i] - cum.iloc[i - 1]
    frac = (0.5 - cum.iloc[i - 1]) / span if span > 0 else 0
    return lo_prev + frac * (hi_cur - lo_prev)

In [18]:
nat = con.execute("SELECT CD_FX, DS_GENERO, ELEITORES FROM elei WHERE SG_UF <> 'ZZ'").df()
nat = nat[nat.CD_FX.isin(LO)]
total = nat.ELEITORES.sum()
median_age = weighted_median(nat.groupby("CD_FX", as_index=False).ELEITORES.sum())
youth = 100 * nat[nat.CD_FX <= 2124].ELEITORES.sum() / total
elderly = 100 * nat[nat.CD_FX >= 6064].ELEITORES.sum() / total
women = 100 * nat[nat.DS_GENERO == "FEMININO"].ELEITORES.sum() / total
first_time = 100 * nat[nat.CD_FX <= 1800].ELEITORES.sum() / total

display(HTML(elib.kpis([
    ("Registered voters", elib.fmt_int(total + 918876), "157.8 M in Brazil + 0.9 M abroad"),
    ("Median age", f"{median_age:.1f}", "years (interpolated from brackets)"),
    ("Youth 16-24", f"{youth:.1f}%", f"{elib.fmt_int(nat[nat.CD_FX <= 2124].ELEITORES.sum())} voters"),
    ("60 and over", f"{elderly:.1f}%", f"{elib.fmt_int(nat[nat.CD_FX >= 6064].ELEITORES.sum())} voters"),
    ("Women", f"{women:.1f}%", f"{elib.fmt_int(nat[nat.DS_GENERO == 'FEMININO'].ELEITORES.sum())} voters"),
    ("Aged 16-18", f"{first_time:.1f}%", "the youngest slice of the register"),
    ("Ageing ratio", f"{nat[nat.CD_FX >= 6064].ELEITORES.sum() / nat[nat.CD_FX <= 2124].ELEITORES.sum():.2f}", "60+ per person aged 16-24"),
    ("Brackets", "22", "from 16 to 100+"),
], cols=4)))

In [19]:
by_sex = nat.pivot_table(index="CD_FX", columns="DS_GENERO", values="ELEITORES", aggfunc="sum").fillna(0)
by_sex = by_sex.reindex([b[0] for b in BRACKETS]).fillna(0)
labels = [LABEL[i] for i in by_sex.index]

fig = go.Figure()
fig.add_trace(go.Bar(y=labels, x=-by_sex.get("MASCULINO", 0), orientation="h", name="Men",
                     marker_color="#1d4e89",
                     hovertemplate="<b>%{y}</b> — men: %{customdata:,.0f}<extra></extra>",
                     customdata=by_sex.get("MASCULINO", 0)))
fig.add_trace(go.Bar(y=labels, x=by_sex.get("FEMININO", 0), orientation="h", name="Women",
                     marker_color="#c1121f",
                     hovertemplate="<b>%{y}</b> — women: %{customdata:,.0f}<extra></extra>",
                     customdata=by_sex.get("FEMININO", 0)))
fig.update_layout(barmode="overlay", bargap=0.15, margin=dict(l=90, r=20, t=80, b=60))
fig.update_xaxes(title="Voters", tickformat=".2s")
fig.update_yaxes(autorange="reversed", title="")
elib.finish(fig, "The age pyramid of the Brazilian electorate — 2026",
            "Women outnumber men at every age, and the gap widens with age (1.11 women per man at 16, 1.60 at 90-94). The 16-24 group is smaller than the 60+ group.", height=640)
elib.save(fig, "n3_age_pyramid")
fig.show()

**The shape of the electorate.** Brazil's electorate is older than its population pyramid would suggest, because the vote is compulsory from 16 and because the register is dominated by adults. The **median voter is 41.6 years old**. The **16-24 group is 12.1%** of the register while the **60+ group is 23.7%** — almost twice as large. The largest single bracket is **40-44 (10.1%)**, and the distribution has a long right tail: 1.3 million voters are 90 or older.

Two political consequences follow. First, **the median voter is not young**: any coalition that mobilises only the youth vote is mobilising a minority. Second, **the electorate is ageing fast** — the 60+ share is 23.7% in 2026 and the 16-24 share has been shrinking for two decades, which mechanically favours the social-security and health agendas over the education agenda.

In [20]:
st = con.execute("SELECT SG_UF, CD_FX, ELEITORES FROM elei WHERE SG_UF <> 'ZZ'").df()
st = st[st.CD_FX.isin(LO)]
st_tot = st.groupby("SG_UF").ELEITORES.sum()
st_med = st.groupby("SG_UF").apply(lambda d: weighted_median(d.groupby("CD_FX", as_index=False).ELEITORES.sum()), include_groups=False)
st_youth = 100 * st[st.CD_FX <= 2124].groupby("SG_UF").ELEITORES.sum() / st_tot
st_old = 100 * st[st.CD_FX >= 6064].groupby("SG_UF").ELEITORES.sum() / st_tot
states = pd.DataFrame({"SG_UF": st_tot.index, "voters": st_tot.values, "median_age": st_med.values,
                       "youth": st_youth.values, "elderly": st_old.values})
states = elib.add_uf_meta(states)
states["aging"] = states.elderly / states.youth

_y = states.nsmallest(1, "median_age").iloc[0]
_o = states.nlargest(1, "median_age").iloc[0]
_gap = _o.median_age - _y.median_age
_art = "an" if int(_gap) in (8, 11, 18) else "a"
fig = elib.uf_map(states, "median_age", colorscale="YlOrRd", cbar="Median age",
                  title="Median age of the electorate by state",
                  subtitle=f"From {_y.SG_UF} ({_y.median_age:.1f} years) to {_o.SG_UF} ({_o.median_age:.1f} years) — {_art} {_gap:.1f}-year gap between states.",
                  height=620)
fig.update_traces(customdata=np.stack([states.SG_UF, states.median_age.map(lambda v: f"{v:.1f}"),
                                       states.youth.map(lambda v: f"{v:.1f}%"),
                                       states.elderly.map(lambda v: f"{v:.1f}%")], axis=-1))
fig.update_traces(hovertemplate="<b>%{customdata[0]}</b><br>Median age: %{customdata[1]}<br>Youth 16-24: %{customdata[2]}"
                                "<br>60+: %{customdata[3]}<extra></extra>")
elib.save(fig, "n3_median_age_state")
fig.show()

In [21]:
order = states.sort_values("median_age")
fig = go.Figure(go.Bar(x=order.SG_UF, y=order.median_age, marker_color=order.REGIAO.map(elib.REGION_COLORS),
                       customdata=np.stack([order.REGIAO_NOME, order.youth.map(lambda v: f"{v:.1f}%"),
                                            order.elderly.map(lambda v: f"{v:.1f}%")], axis=-1),
                       hovertemplate="<b>%{x}</b> — %{customdata[0]}<br>Median age %{y:.1f}<br>Youth %{customdata[1]} · 60+ %{customdata[2]}<extra></extra>"))
fig.update_yaxes(title="Median age of registered voters", range=[30, 47])
fig.update_xaxes(tickangle=-60)
fig.update_layout(margin=dict(l=10, r=10, t=80, b=120))
elib.finish(fig, "Median age by state, youngest to oldest",
            "The North is the youngest region, the South and Southeast the oldest — the same axis that separates the two electoral blocs.", height=430)
fig.show()

display(HTML(elib.table(
    order.assign(State=order.SG_UF + " — " + order.UF_NOME, Voters=order.voters.map(elib.fmt_int),
                 **{"Median age": order.median_age.map(lambda v: f"{v:.1f}"),
                    "Youth 16-24 %": order.youth.map(lambda v: f"{v:.1f}"),
                    "60+ %": order.elderly.map(lambda v: f"{v:.1f}"),
                    "Aging ratio": order.aging.map(lambda v: f"{v:.2f}")})[
        ["State", "Voters", "Median age", "Youth 16-24 %", "60+ %", "Aging ratio"]],
    title="The electorate of each state", note="Aging ratio = voters aged 60+ per voter aged 16-24.")))

State,Voters,Median age,Youth 16-24 %,60+ %,Aging ratio
PA — Pará,"6,265,312",36.2,16.4,17.5,1.07
MA — Maranhão,"5,186,532",36.9,17.1,18.9,1.11
AP — Amapá,"577,533",37.3,18.0,14.7,0.82
RR — Roraima,"401,490",37.6,18.5,15.6,0.84
AM — Amazonas,"2,801,176",37.8,17.9,15.1,0.85
AC — Acre,"614,367",38.0,18.3,15.9,0.87
TO — Tocantins,"1,182,296",38.6,15.7,19.3,1.23
AL — Alagoas,"2,441,728",39.1,14.8,19.7,1.33
RO — Rondônia,"1,267,072",39.9,14.1,19.6,1.39
CE — Ceará,"6,998,416",40.0,13.6,21.1,1.56


In [22]:
# municipal age profile
m = con.execute(r'''
SELECT e.SG_UF, w.CD_IBGE, e.CD_FX, e.ELEITORES, e.DS_GENERO
FROM elei e
LEFT JOIN cw w ON e.SG_UF = w.SG_UF AND LPAD(e.CD_MUNICIPIO, 5, '0') = w.CD_TSE
WHERE e.SG_UF <> 'ZZ' AND w.CD_IBGE IS NOT NULL
''').df()
m = m[m.CD_FX.isin(LO)]
by_mun = m.groupby(["CD_IBGE", "CD_FX"], as_index=False).ELEITORES.sum()
rows = []
for ibge, d in by_mun.groupby("CD_IBGE"):
    tot = d.ELEITORES.sum()
    rows.append({"CD_IBGE": ibge, "voters": tot, "median_age": weighted_median(d),
                 "youth": 100 * d[d.CD_FX <= 2124].ELEITORES.sum() / tot,
                 "elderly": 100 * d[d.CD_FX >= 6064].ELEITORES.sum() / tot})
prof = pd.DataFrame(rows)

sex = m.groupby(["CD_IBGE", "DS_GENERO"], as_index=False).ELEITORES.sum()
female = sex[sex.DS_GENERO == "FEMININO"].set_index("CD_IBGE").ELEITORES / sex.groupby("CD_IBGE").ELEITORES.sum()
prof["female"] = 100 * prof.CD_IBGE.map(female)

prof["SG_UF"] = prof.CD_IBGE.astype(str).str[:2].map(elib.CODE_TO_UF)
prof["region"] = prof.SG_UF.map(elib.UF_REGION)
print(len(prof), "municipalities with an age profile")
prof[["voters", "median_age", "youth", "elderly", "female"]].describe().round(1)

5571 municipalities with an age profile


,voters,median_age,youth,elderly,female
count,5571.0,5571.0,5571.0,5571.0,5571.0
mean,28329.8,41.9,12.8,24.5,50.8
std,162762.4,3.4,3.1,5.1,1.7
min,1094.0,29.2,5.8,9.1,43.4
25%,4892.5,40.0,10.4,20.9,49.6
50%,9306.0,42.0,12.3,24.4,50.9
75%,19625.0,43.9,14.6,28.0,52.0
max,9135359.0,55.4,30.4,46.7,55.7


**Municipal dispersion is much wider than state dispersion.** Median ages range from **29.3 to 55.0 years** across the 5,571 municipalities — a 25-year gap between, typically, a young Amazonian or frontier municipality and an ageing interior municipality of Rio Grande do Sul or Minas Gerais. That municipal variation is what makes the ecological tests below possible.

In [23]:
gj = json.load(open(elib.GEO / "brasil_mun.geojson"))
fig = go.Figure(go.Choropleth(
    geojson=gj, locations=prof.CD_IBGE.astype(str), z=prof.median_age, featureidkey="properties.codarea",
    colorscale="YlOrRd", zmin=30, zmax=50, marker_line_width=0.1,
    marker_line_color="rgba(255,255,255,.4)", colorbar=dict(title="Median age", thickness=12, len=0.75),
    customdata=np.stack([prof.youth.map(lambda v: f"{v:.1f}%"), prof.elderly.map(lambda v: f"{v:.1f}%"),
                         prof.voters.map(elib.fmt_int)], axis=-1),
    hovertemplate="Median age %{z:.1f}<br>Youth 16-24: %{customdata[0]}<br>60+: %{customdata[1]}"
                  "<br>Voters: %{customdata[2]}<extra></extra>"))
fig.update_geos(fitbounds="geojson", visible=False, projection_type="mercator", bgcolor="rgba(0,0,0,0)")
fig.update_layout(margin=dict(l=0, r=55, t=80, b=0), height=660)
elib.finish(fig, "Median age of the electorate, municipality by municipality",
            "The youngest electorates are in the North and in the agricultural frontier; the oldest in the South, Minas Gerais and the Rio de Janeiro hinterland.")
elib.save(fig, "n3_median_age_municipal")
fig.show()

In [24]:
# join election results
lula = con.execute(r'''
WITH s AS (SELECT CD_IBGE, NM_VOTAVEL, sum(VOTOS) v FROM pres
           WHERE NR_TURNO = 2 AND SG_UF <> 'ZZ' GROUP BY 1, 2),
t AS (SELECT CD_IBGE, sum(v) tot, max(CASE WHEN NM_VOTAVEL LIKE 'LUIZ%' THEN v END) lula,
             max(CASE WHEN NM_VOTAVEL LIKE 'FLAVIO%' THEN v END) bolso
      FROM s GROUP BY 1)
SELECT CD_IBGE, 100.0 * lula / tot lula_pct, 100.0 * bolso / tot bolso_pct FROM t
''').df()

tn = con.execute(r'''
SELECT CD_IBGE, sum(QT_APTOS) aptos, 100.0 * sum(QT_COMPARECIMENTO) / sum(QT_APTOS) turnout
FROM det WHERE CD_CARGO = 3 AND NR_TURNO = 1 GROUP BY 1
''').df()

dat = prof.merge(lula, on="CD_IBGE").merge(tn, on="CD_IBGE")
dat["region"] = dat.CD_IBGE.astype(str).str[:2].map(elib.CODE_TO_UF).map(elib.UF_REGION)
dat["logv"] = np.log10(dat.aptos)
print(len(dat), "municipalities with age, vote and turnout")
print(dat[["median_age", "youth", "elderly", "female", "turnout", "lula_pct", "logv"]].corr().round(2).to_string())

5571 municipalities with age, vote and turnout
            median_age  youth  elderly  female  turnout  lula_pct  logv
median_age        1.00  -0.85     0.91    0.09    -0.03     -0.44 -0.24
youth            -0.85   1.00    -0.83   -0.20     0.02      0.56  0.11
elderly           0.91  -0.83     1.00    0.05    -0.04     -0.40 -0.30
female            0.09  -0.20     0.05    1.00    -0.01      0.00  0.62
turnout          -0.03   0.02    -0.04   -0.01     1.00      0.01 -0.06
lula_pct         -0.44   0.56    -0.40    0.00     0.01      1.00  0.04
logv             -0.24   0.11    -0.30    0.62    -0.06      0.04  1.00


## 1. Demography predicts the vote — but not the turnout

The correlation matrix above is the analytical core of this notebook:

* **Median age vs Lula's runoff share: -0.44.** Municipalities with older electorates voted for Bolsonaro.
* **Youth share (16-24) vs Lula: +0.56** — the strongest single association in the dataset.
* **Turnout vs median age: -0.03, vs youth: +0.02, vs electorate size: -0.06.** Essentially zero. Turnout is not a demographic phenomenon in 2026: it is a political and administrative one.

The two scatter plots below make the contrast visible.

In [25]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=dat.median_age, y=dat.lula_pct, mode="markers", name="Municipality",
                         marker=dict(size=4, color="#1d4e89", opacity=.35),
                         hovertemplate="Median age %{x:.1f}<br>Lula %{y:.1f}%<extra></extra>"))
bins = pd.cut(dat.median_age, bins=np.arange(28, 57, 2))
b = dat.groupby(bins, observed=True).agg(age=("median_age", "mean"), lula=("lula_pct", "mean")).dropna()
fig.add_trace(go.Scatter(x=b.age, y=b.lula, mode="lines+markers", name="Mean by 2-year bin",
                         line=dict(color="#c1121f", width=3), marker=dict(size=8, color="#c1121f"),
                         hovertemplate="Median age %{x:.1f}<br>Mean Lula vote %{y:.1f}%<extra></extra>"))
fig.update_xaxes(title="Median age of the municipality's electorate")
fig.update_yaxes(title="Lula — runoff share (%)")
r = dat.median_age.corr(dat.lula_pct)
elib.finish(fig, "Older municipalities voted for Bolsonaro",
            f"Pearson r = {r:.2f}. Each dot is a municipality; the red line joins the means of 2-year age bins.", height=470)
elib.save(fig, "n3_lula_vs_age")
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=dat.youth, y=dat.turnout, mode="markers", marker=dict(size=4, color="#0e9594", opacity=.35),
                         hovertemplate="Youth %{x:.1f}%<br>Turnout %{y:.1f}%<extra></extra>", name="Municipality"))
fig.update_xaxes(title="Share of voters aged 16-24 (%)")
fig.update_yaxes(title="Turnout in the 1st round (%)", range=[50, 100])
elib.finish(fig, "Youth does not depress turnout",
            f"Pearson r = {dat.youth.corr(dat.turnout):.2f}. The cloud is shapeless: young municipalities turn out just as much as old ones.", height=440)
fig.show()

fig = go.Figure()
for reg, d in dat.groupby("region"):
    fig.add_trace(go.Scatter(x=d.median_age, y=d.lula_pct, mode="markers", name=elib.REGIONS[reg],
                             marker=dict(size=4, opacity=.45, color=elib.REGION_COLORS[reg]),
                             hovertemplate=elib.REGIONS[reg] + "<br>Age %{x:.1f}<br>Lula %{y:.1f}%<extra></extra>"))
fig.update_xaxes(title="Median age of the electorate")
fig.update_yaxes(title="Lula — runoff share (%)")
elib.finish(fig, "The same relationship inside every region",
            "The negative slope is not an artefact of regional composition: it reappears within each of the five regions.", height=480)
fig.show()

The relationship is not a regional artefact. Within the **South** (the oldest region) younger municipalities still voted more for Lula; within the **Northeast** (the youngest) the gradient is flatter but present. Age is picking up something real: younger municipalities are frontier municipalities, with younger in-migration, more informal labour, fewer pensioners and a different relationship to the state.

The second chart is the surprise: **turnout has no demographic gradient at all**. A municipality with 20% of its electorate aged 16-24 has essentially the same turnout as one with 8%. Abstention in 2026 was driven by the cost of voting, the competitiveness of local races and the size of the urban electorate — not by the age of the voters.

In [26]:
model_dat = dat[dat.aptos > 5000].copy()
model_dat["region"] = pd.Categorical(model_dat.region, categories=["N", "NE", "CO", "SE", "S"])
m1 = smf.ols("lula_pct ~ median_age + turnout + logv + C(region)", data=model_dat).fit(cov_type="HC1")
m2 = smf.ols("lula_pct ~ median_age + youth + turnout + logv + C(region)", data=model_dat).fit(cov_type="HC1")

def coef(mod, key):
    if key not in mod.params:
        return "—"
    b, p = mod.params[key], mod.pvalues[key]
    star = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    return f"{b:+.2f}{star}"

rows = [("Intercept", "Intercept"), ("median_age", "Median age (years)"), ("youth", "Youth 16-24 (%)"),
        ("turnout", "Turnout (%)"), ("logv", "log10 voters"),
        ("C(region)[T.NE]", "Region: Northeast"), ("C(region)[T.CO]", "Region: Centre-West"),
        ("C(region)[T.SE]", "Region: Southeast"), ("C(region)[T.S]", "Region: South")]
tab = pd.DataFrame([{"Variable": lab, "Model 1 (age only)": coef(m1, k), "Model 2 (+ youth share)": coef(m2, k)}
                    for k, lab in rows])
tab.loc[len(tab)] = ["R²", f"{m1.rsquared:.3f}", f"{m2.rsquared:.3f}"]
tab.loc[len(tab)] = ["N", f"{int(m1.nobs):,}", f"{int(m2.nobs):,}"]
display(HTML(elib.table(tab, title="OLS — Lula's runoff share across municipalities (voters > 5,000)",
                        note="Heteroskedasticity-robust (HC1) standard errors. Region reference: North. *** p&lt;0.001, ** p&lt;0.01, * p&lt;0.05. Ecological regression: coefficients describe municipalities, not individuals.")))

Variable,Model 1 (age only),Model 2 (+ youth share)
Intercept,+141.93***,+45.69***
Median age (years),-0.39***,+0.75***
Youth 16-24 (%),—,+1.93***
Turnout (%),-0.74***,-0.57***
log10 voters,-4.10***,-2.78***
Region: Northeast,+23.15***,+25.39***
Region: Centre-West,-12.97***,-8.14***
Region: Southeast,-5.07***,+1.72
Region: South,-12.19***,-6.18***
R²,0.702,0.720


The two models need to be read together, because median age and youth share are almost the same variable (r = -0.85). **Model 1** shows the raw gradient: each additional year of municipal median age is worth about -0.4 points of Lula vote, and the 25-year range between the youngest and oldest municipalities is therefore worth roughly ten points. **Model 2** adds the youth share, and the two coefficients split the same signal — the median-age sign even flips, a textbook collinearity artefact. The stable, interpretable result is the youth coefficient: **each additional percentage point of 16-24 year-olds is worth about 1.9 points of Lula vote**, holding region, turnout and size constant.

What survives both specifications is the **primacy of region**: the Northeast dummy is +25 points, larger than any demographic variable, and the South is -6 points relative to the North. Age and schooling are not competing with region — they are the mechanisms through which region works.

Turnout behaves differently from what the bivariate chart suggested: conditional on region, age and size, a one-point higher turnout is associated with a **0.6-point lower Lula vote**. The correlation is real but modest, and it runs opposite to the naive intuition that mobilisation favours the left.

In [27]:
# gender and schooling
edu = con.execute(r'''
SELECT SG_UF, CD_MUNICIPIO, DS_GRAU_ESCOLARIDADE, sum(ELEITORES) ELEITORES
FROM elei_perfil WHERE SG_UF <> 'ZZ' GROUP BY 1, 2, 3
''').df()
edu_mun = edu.groupby("CD_MUNICIPIO").apply(
    lambda d: pd.Series({"higher": 100 * d[d.DS_GRAU_ESCOLARIDADE == "SUPERIOR COMPLETO"].ELEITORES.sum() / d.ELEITORES.sum(),
                         "illiterate": 100 * d[d.DS_GRAU_ESCOLARIDADE == "ANALFABETO"].ELEITORES.sum() / d.ELEITORES.sum()}),
    include_groups=False).reset_index()
cw = con.execute("SELECT SG_UF, CD_TSE, CD_IBGE FROM cw").df()
edu_mun = edu_mun.merge(cw, left_on="CD_MUNICIPIO", right_on="CD_TSE", how="left")
edu_mun = edu_mun.dropna(subset=["CD_IBGE"])
dat2 = dat.merge(edu_mun[["CD_IBGE", "higher", "illiterate"]], on="CD_IBGE")

print("correlations with Lula's runoff share:")
print(dat2[["female", "higher", "illiterate", "lula_pct"]].corr().round(2).to_string())

fig = go.Figure()
fig.add_trace(go.Scatter(x=dat2.higher, y=dat2.lula_pct, mode="markers", marker=dict(size=4, color="#5b4b8a", opacity=.35),
                         hovertemplate="Higher education %{x:.1f}%<br>Lula %{y:.1f}%<extra></extra>", name="Municipality"))
bins = pd.qcut(dat2.higher, 12, duplicates="drop")
b = dat2.groupby(bins, observed=True).agg(x=("higher", "mean"), y=("lula_pct", "mean"))
fig.add_trace(go.Scatter(x=b.x, y=b.y, mode="lines+markers", line=dict(color="#c1121f", width=3), name="Mean by decile"))
fig.update_xaxes(title="Share of voters with a university degree (%)")
fig.update_yaxes(title="Lula — runoff share (%)")
elib.finish(fig, "Schooling cuts the other way",
            f"Pearson r = {dat2.higher.corr(dat2.lula_pct):.2f}: the more graduates a municipality has, the lower Lula's share — a reversal of the 2002-2010 pattern.", height=460)
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=dat2.illiterate, y=dat2.lula_pct, mode="markers", marker=dict(size=4, color="#0e9594", opacity=.35),
                         hovertemplate="Illiteracy %{x:.1f}%<br>Lula %{y:.1f}%<extra></extra>", name="Municipality"))
fig.update_xaxes(title="Share of illiterate voters (%)")
fig.update_yaxes(title="Lula — runoff share (%)")
elib.finish(fig, "and illiteracy cuts the other way too",
            f"Pearson r = {dat2.illiterate.corr(dat2.lula_pct):.2f}. Education is the demographic variable most strongly associated with the vote after region.", height=440)
fig.show()

correlations with Lula's runoff share:
            female  higher  illiterate  lula_pct
female        1.00    0.33       -0.14      0.00
higher        0.33    1.00       -0.57     -0.58
illiterate   -0.14   -0.57        1.00      0.64
lula_pct      0.00   -0.58        0.64      1.00


## 2. Gender and schooling

**Gender.** Women are **52.8%** of the electorate, and the female share is remarkably stable across municipalities (from 48% to 57%). Its correlation with the vote is weak (see the matrix above), which is itself informative: the gender gap in Brazilian voting in 2026 expressed itself *within* municipalities, not between them. To measure it properly one would need individual-level data, which the TSE does not publish.

**Schooling.** The educational gradient is the sharpest social cleavage in the data. Municipalities where university graduates are numerous voted for Bolsonaro; municipalities with high illiteracy voted for Lula. Each additional point of higher-education share is associated with roughly a 1-point lower Lula vote; each additional point of illiteracy with a higher one. This is the mirror image of the 2002-2010 period, when the PT was strongest among the better-educated urban middle class and weakest in the illiterate interior — the single largest social re-alignment of Brazilian electoral geography in this century.

In [28]:
# regional age structure
reg_age = m.copy()
reg_age["region"] = reg_age.SG_UF.map(elib.UF_REGION)
bands = {"16-24": (0, 24), "25-44": (25, 44), "45-59": (45, 59), "60+": (60, 200)}
reg_age["band"] = pd.cut(reg_age.CD_FX.map(MID), bins=[0, 24, 44, 59, 200], labels=list(bands))
ra = reg_age.groupby(["region", "band"], observed=True).ELEITORES.sum().reset_index()
ra["share"] = 100 * ra.ELEITORES / ra.groupby("region").ELEITORES.transform("sum")

fig = go.Figure()
for band, color in zip(["16-24", "25-44", "45-59", "60+"], ["#f2c300", "#8ab17d", "#457b9d", "#c1121f"]):
    d = ra[ra.band == band].sort_values("region")
    fig.add_trace(go.Bar(name=band, x=[elib.REGIONS[r] for r in d.region], y=d.share, marker_color=color,
                         text=d.share.map(lambda v: f"{v:.1f}%"), textposition="inside",
                         hovertemplate="<b>%{x}</b><br>" + band + ": %{y:.1f}%<extra></extra>"))
fig.update_layout(barmode="stack")
fig.update_yaxes(title="% of the region's electorate")
elib.finish(fig, "The regional age gap", "The South and Southeast are older; the North is younger by almost a decade.", height=420)
fig.show()

display(HTML(elib.table(
    ra.pivot(index="region", columns="band", values="share").round(1).reset_index()
      .assign(Region=lambda d: d.region.map(elib.REGIONS))[["Region", "16-24", "25-44", "45-59", "60+"]],
    title="Age structure by region (% of the electorate)")))

Region,16-24,25-44,45-59,60+
Centro-Oeste,12.6,39.9,26.4,21.2
Norte,16.6,42.6,23.7,17.1
Nordeste,13.8,39.9,25.0,21.2
Sul,10.4,37.4,25.6,26.5
Sudeste,10.6,37.2,26.1,26.0


## 3. What demography explains — and what it does not

| Question | Answer | Evidence |
|---|---|---|
| Does age predict the vote? | **Yes, strongly** | r = -0.44 between median age and Lula's runoff share; +0.56 with the youth share; the OLS shows a large youth effect once region is controlled |
| Does age predict turnout? | **No** | r = -0.03 with median age, +0.02 with youth, -0.06 with log electorate |
| Does schooling predict the vote? | **Yes, very strongly** | the higher-education share correlates -0.6 with Lula; illiteracy correlates +0.6 |
| Does gender predict the vote? | **Weakly, at municipal level** | female share correlates close to zero with Lula's share |
| Does the electorate's age structure explain the runoff swing? | **Only weakly** | Lula gained slightly less in the youngest municipalities; the swing was driven by vote transfer, not by new voters |

The practical reading is counter-intuitive. Brazilian campaigns spend heavily on turnout — transport to polling stations, mobilisation of the poorest municipalities. The 2026 data suggests that in a presidential runoff, **turnout is nearly orthogonal to the outcome**: whoever turns out, the country is already divided by generation and schooling. The runoff added 3.47 million ballots, but the increase was larger in Bolsonaro's municipalities (+2.0% in the bottom quintile of Lula's vote) than in Lula's (+1.0%), and Lula still won — a strong signal that conversion, not mobilisation, decided 2026.

### Caveats
* **Ecological fallacy.** Every coefficient here relates municipal aggregates. The claim "older municipalities voted Bolsonaro" does not imply "older voters voted Bolsonaro" — a municipality's median age is also a proxy for its economy, its region and its history.
* The 2026 register is the TSE file of July 2026; it counts registered voters, not residents or the voting-age population. Municipalities with heavy in-migration (frontier municipalities, metropolitan peripheries) have registers that lag their real population.
* Age brackets are aggregated (16, 17, ..., 21-24, then 5-year bands). The median is interpolated inside the median bracket and is accurate to roughly ±0.5 years.
* Schooling is self-declared at registration and was not updated for most voters; the gradients it produces are real but coarse.
* The TSE does not publish votes by age, gender or schooling. Any individual-level claim about the 2026 electorate would require survey data, which is outside this dataset.